# HawkEars Detection Validation Interface

An interactive Jupyter notebook for manually validating [HawkEars](https://github.com/jhuus/HawkEars) acoustic detections.

**Features:**
- Stratified sampling across confidence score bins
- Dual-channel spectrogram display (left/right microphone)
- Audio playback controls
- Keyboard shortcuts for fast validation (T/F/U/W/B/S)
- Progress saving and session resumption
- Configurable spectrogram appearance (colormap, dynamic range)

**Workflow:**
1. Point to your HawkEars output CSV and audio directories
2. Configure species and confidence threshold
3. Generate a stratified validation sample
4. Run the interactive validation interface
5. Export validated results for analysis

## Requirements

```
pip install pandas numpy librosa matplotlib sounddevice ipywidgets
```

Tested with JupyterLab and VS Code notebooks. Requires `ipywidgets` support in your notebook environment.

## 1. Setup and Configuration

Edit the cell below to match your project. All user-configurable settings are here.

In [ ]:
import pandas as pd
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import sounddevice as sd
from pathlib import Path
import warnings
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output, Audio
from matplotlib.colors import LinearSegmentedColormap, PowerNorm

from utils import (
    parse_audio_filename,
    build_audio_file_lookup,
    load_audio_stereo,
    play_audio,
)

warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
# =============================================================================
# USER CONFIGURATION - Edit these settings for your project
# =============================================================================

# Path to your merged HawkEars output CSV.
# Expected columns: filename, species, confidence, start_time, end_time,
#                    location, recording_datetime
HAWKEARS_CSV = "path/to/your/hawkears_labels.csv"

# Directories containing your audio (.wav) files.
# The notebook will recursively search these for .wav files.
AUDIO_DIRS = [
    r"path/to/audio/directory_1",
    r"path/to/audio/directory_2",
]

# Species codes to validate. Set to None to auto-detect all species in the CSV.
FOCAL_SPECIES = None  # e.g. ['OVEN', 'BRCR'] or None for all species

# Confidence score threshold - only validate detections at or above this score
SCORE_THRESHOLD = 0.1

# Number of detections to sample PER SPECIES
VALIDATION_SAMPLES = 100

# Output directory for validation files (created automatically)
OUTPUT_DIR = "validation_output"

# =============================================================================
# OPTIONAL - Filename parsing
# =============================================================================
# HawkEars output filenames typically follow: LocationID_YYYYMMDD_HHMMSS_HawkEars.txt
# The audio .wav files follow:               LocationID_YYYYMMDD_HHMMSS.wav
#
# If your filenames use a different delimiter or structure, edit
# parse_audio_filename() in the next section.

# Characters in filenames to replace with '_' before parsing
FILENAME_SANITIZE_CHARS = ['$']

# Prefixes to strip from filenames before parsing (e.g. channel prefixes)
FILENAME_STRIP_PREFIXES = ['0+1_']

In [ ]:
# Create output directory
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Derived file paths
PROGRESS_FILE = str(Path(OUTPUT_DIR) / "validation_progress.csv")
RESULTS_FILE = str(Path(OUTPUT_DIR) / "validation_results.csv")
FILE_LOOKUP_FILE = str(Path(OUTPUT_DIR) / "audio_file_lookup.csv")
SAMPLE_FILE = str(Path(OUTPUT_DIR) / "validation_sample.csv")

print(f"Output directory: {OUTPUT_DIR}")
print(f"Progress file:   {PROGRESS_FILE}")
print(f"Results file:    {RESULTS_FILE}")
print(f"File lookup:     {FILE_LOOKUP_FILE}")

## 2. Build Audio File Lookup

Searches your audio directories and builds a lookup table mapping HawkEars detection filenames to audio file paths on disk. This is cached after the first run.

In [ ]:
# Build or load lookup table
audio_lookup = build_audio_file_lookup(
    AUDIO_DIRS, FILE_LOOKUP_FILE,
    sanitize_chars=FILENAME_SANITIZE_CHARS,
    strip_prefixes=FILENAME_STRIP_PREFIXES
)
print(f"
Audio lookup: {len(audio_lookup)} files")
audio_lookup.head()

## 3. Load and Merge HawkEars Detections

Loads the HawkEars output CSV and joins it with the audio file lookup so each detection has a path to its source audio.

In [ ]:
# Load HawkEars output
detections = pd.read_csv(HAWKEARS_CSV)

# Clean column names (remove BOM if present)
detections.columns = detections.columns.str.replace('\ufeff', '')

# Derive filename_base from the HawkEars output filename column
detections['filename_base'] = detections['filename'].str.replace(
    '_HawkEars.txt', '', regex=False
)

# Auto-detect species if not specified
if FOCAL_SPECIES is None:
    FOCAL_SPECIES = sorted(detections['species'].unique().tolist())
    print(f"Auto-detected {len(FOCAL_SPECIES)} species: {FOCAL_SPECIES}")
else:
    print(f"Focal species: {FOCAL_SPECIES}")

# Filter to focal species
detections = detections[detections['species'].isin(FOCAL_SPECIES)].copy()
print(f"Detections for focal species: {len(detections)}")

# Merge with audio file lookup
detections = detections.merge(
    audio_lookup[['location', 'recording_datetime', 'filename_base', 'file_path']],
    on=['location', 'recording_datetime', 'filename_base'],
    how='left'
)

n_missing = detections['file_path'].isna().sum()
print(f"Detections with missing audio files: {n_missing}")

print(f"\nDetections per species:")
print(detections['species'].value_counts())

## 4. Create Validation Sample

Samples detections stratified by confidence score bins (0.1 width). When bins have fewer samples than needed, extras are drawn from higher-confidence bins.

If you have previous validation results, uncomment `exclude_already_validated()` to avoid re-validating the same detections.

In [ ]:
def exclude_already_validated(df, results_file=RESULTS_FILE):
    """
    Remove detections that have already been validated from the sampling pool.

    Parameters:
        df:           Detection DataFrame
        results_file: Path to existing validation results CSV

    Returns:
        Filtered DataFrame with validated detections removed
    """
    if not Path(results_file).exists():
        print(f"No validation results file found at {results_file}")
        print("All detections are available for sampling.")
        return df

    validated_df = pd.read_csv(results_file)
    key_cols = ['filename', 'start_time', 'end_time', 'species']

    missing_cols = [c for c in key_cols if c not in df.columns]
    if missing_cols:
        print(f"Warning: Missing columns in detection data: {missing_cols}")
        return df

    missing_in_results = [c for c in key_cols if c not in validated_df.columns]
    if missing_in_results:
        print(f"Warning: Missing columns in results file: {missing_in_results}")
        return df

    df = df.copy()
    df['_match_key'] = (
        df['filename'].astype(str) + '_' +
        df['start_time'].astype(str) + '_' +
        df['end_time'].astype(str) + '_' +
        df['species'].astype(str)
    )
    validated_df['_match_key'] = (
        validated_df['filename'].astype(str) + '_' +
        validated_df['start_time'].astype(str) + '_' +
        validated_df['end_time'].astype(str) + '_' +
        validated_df['species'].astype(str)
    )

    validated_keys = set(validated_df['_match_key'])
    original_count = len(df)
    df_filtered = df[~df['_match_key'].isin(validated_keys)].copy()
    df_filtered = df_filtered.drop(columns=['_match_key'])
    excluded_count = original_count - len(df_filtered)

    print(f"Already validated: {len(validated_keys)}")
    print(f"Excluded from sampling: {excluded_count}")
    print(f"Available for sampling: {len(df_filtered)}")

    return df_filtered


# Uncomment to exclude previously validated detections:
# detections = exclude_already_validated(detections)

In [ ]:
def create_validation_sample(df, score_threshold, n_samples, focal_species):
    """
    Create a stratified validation sample from detections above a confidence threshold.

    Samples are allocated evenly across confidence score bins of 0.1 width.
    When bins have insufficient samples, extras are drawn from higher-confidence
    bins (0.7+), then from any remaining unsampled detections.

    Parameters:
        df:              Full detection DataFrame
        score_threshold: Minimum confidence score to include
        n_samples:       Number of samples per species
        focal_species:   List of species codes (validation order)

    Returns:
        DataFrame of sampled detections with a validation_id column
    """
    df_filtered = df[df['confidence'] >= score_threshold].copy()
    print(f"Detections above {score_threshold} threshold: {len(df_filtered)}")
    print(f"\nDetections by species (above threshold):")
    print(df_filtered['species'].value_counts())

    bins = np.arange(0.1, 1.1, 0.1)
    bin_labels = [f"{bins[i]:.1f}-{bins[i+1]:.1f}" for i in range(len(bins)-1)]

    validation_samples = []

    for species in focal_species:
        sp_data = df_filtered[df_filtered['species'] == species].copy()
        print(f"\n{species}: {len(sp_data)} detections above threshold")

        if len(sp_data) == 0:
            print(f"  Warning: No detections for {species}")
            continue

        sp_data['confidence_bin'] = pd.cut(
            sp_data['confidence'], bins=bins, labels=bin_labels, include_lowest=True
        )

        bin_counts = sp_data['confidence_bin'].value_counts()
        n_bins_with_data = len(bin_counts[bin_counts > 0])

        if n_bins_with_data == 0:
            print(f"  Warning: No valid bins for {species}")
            continue

        samples_per_bin = n_samples // n_bins_with_data
        remaining_samples = n_samples % n_bins_with_data
        print(f"  {n_bins_with_data} bins with data, {samples_per_bin} samples/bin (+{remaining_samples} extra)")

        bin_samples = []
        extra_allocated = 0
        sampled_ids = set()

        for bin_label in bin_labels:
            bin_data = sp_data[sp_data['confidence_bin'] == bin_label]
            n_available = len(bin_data)
            if n_available == 0:
                continue

            n_from_bin = samples_per_bin
            if extra_allocated < remaining_samples:
                n_from_bin += 1
                extra_allocated += 1

            if n_from_bin > n_available:
                print(f"    {bin_label}: {n_available}/{n_available} (all available, needed {n_from_bin})")
                bin_sample = bin_data.copy()
            else:
                print(f"    {bin_label}: {n_from_bin}/{n_available} available")
                bin_sample = bin_data.sample(n=n_from_bin, random_state=42, replace=False)

            sampled_ids.update(bin_sample.index)
            bin_samples.append(bin_sample)

        current_count = sum(len(s) for s in bin_samples)
        shortage = n_samples - current_count

        # Fill shortage from high-confidence bins
        if shortage > 0:
            high_conf_bins = ['0.7-0.8', '0.8-0.9', '0.9-1.0']
            high_conf_data = sp_data[
                (sp_data['confidence_bin'].isin(high_conf_bins)) &
                (~sp_data.index.isin(sampled_ids))
            ]
            if len(high_conf_data) > 0:
                n_extra = min(shortage, len(high_conf_data))
                extra_sample = high_conf_data.sample(n=n_extra, random_state=42, replace=False)
                sampled_ids.update(extra_sample.index)
                bin_samples.append(extra_sample)
                shortage -= n_extra

            # Still short - sample from any remaining
            if shortage > 0:
                remaining_data = sp_data[~sp_data.index.isin(sampled_ids)]
                if len(remaining_data) > 0:
                    n_extra = min(shortage, len(remaining_data))
                    extra_sample = remaining_data.sample(n=n_extra, random_state=42, replace=False)
                    sampled_ids.update(extra_sample.index)
                    bin_samples.append(extra_sample)

        if bin_samples:
            species_sample = pd.concat(bin_samples, ignore_index=False)
            species_sample = species_sample.drop_duplicates()
            species_sample = species_sample.drop(columns=['confidence_bin'])
            validation_samples.append(species_sample)
            print(f"  Final sample for {species}: {len(species_sample)}/{n_samples}")

    validation_df = pd.concat(validation_samples, ignore_index=True)

    # Sort by species in order of focal_species list
    species_order = {sp: i for i, sp in enumerate(focal_species)}
    validation_df['species_order'] = validation_df['species'].map(species_order)
    validation_df = validation_df.sort_values('species_order').reset_index(drop=True)
    validation_df = validation_df.drop(columns=['species_order'])

    validation_df['validation_id'] = range(len(validation_df))

    return validation_df


# Create validation sample
validation_sample = create_validation_sample(
    detections, SCORE_THRESHOLD, VALIDATION_SAMPLES, FOCAL_SPECIES
)

print(f"\n{'='*60}")
print(f"Total validation sample: {len(validation_sample)}")
print(f"{'='*60}")
print(validation_sample['species'].value_counts())

# Save sample to disk
validation_sample.to_csv(SAMPLE_FILE, index=False)
print(f"\nSample saved to {SAMPLE_FILE}")

## 5. Run Validation Interface

Execute the cell below to start the interactive validation session.

**Keyboard shortcuts:**
| Key | Action |
|-----|--------|
| T | True Positive |
| F | False Positive |
| U | Uncertain |
| W | Bad Weather / Noise |
| B | Go back to previous detection |
| S | Skip (moves detection to end of queue) |

Progress is auto-saved every 20 validations. Use **Save Progress** to save manually. If you close and re-run this cell, the session resumes where you left off.

In [ ]:
# Spectrogram display settings
DEFAULT_DYNAMIC_RANGE = 65  # dB - lower = more contrast

# Yellow-red colormap (alternative to grayscale)
colors = ['#FFFF00', '#FFD700', '#FFA500', '#FF4500', '#FF0000', '#8B0000']
yellow_red_cmap = LinearSegmentedColormap.from_list('yellow_red', colors, N=256)


class ValidationInterface:
    def __init__(self, validation_df, progress_file=PROGRESS_FILE,
                 results_file=RESULTS_FILE, colormap='bw'):
        """
        Interactive validation interface for acoustic detections.

        Parameters:
            validation_df:  DataFrame of detections to validate
            progress_file:  CSV file for saving/loading progress
            results_file:   CSV file for final results
            colormap:       'bw' for grayscale or 'color' for yellow-red
        """
        self.detections = validation_df.copy()
        self.progress_file = progress_file
        self.results_file = results_file
        self.current_idx = 0
        self.results = []
        self.total_detections = len(validation_df)

        self.cmap = 'gray_r' if colormap == 'bw' else yellow_red_cmap
        self.dynamic_range = DEFAULT_DYNAMIC_RANGE

        n_missing = validation_df['file_path'].isna().sum()
        if n_missing > 0:
            print(f"Warning: {n_missing} detections have missing file paths (auto-skipped)")

        self._load_progress()
        self._create_widgets()

        self.current_audio_left = None
        self.current_audio_right = None
        self.current_sr = None

    def _load_progress(self):
        """Resume from a previous validation session."""
        if Path(self.progress_file).exists():
            progress_df = pd.read_csv(self.progress_file)
            self.results = progress_df.to_dict('records')

            validated_ids = set(progress_df['validation_id'])
            all_ids = set(self.detections['validation_id'])
            remaining_ids = all_ids - validated_ids

            self.detections = self.detections[
                self.detections['validation_id'].isin(remaining_ids)
            ].reset_index(drop=True)
            self.current_idx = 0

            print(f"Resuming: {len(validated_ids)} done, {len(remaining_ids)} remaining")

    def _create_widgets(self):
        """Create UI widgets."""
        def btn(desc, style, w='200px'):
            return widgets.Button(
                description=desc, button_style=style,
                layout=widgets.Layout(width=w, height='50px')
            )

        self.button_tp = btn('True Positive (T)', 'success')
        self.button_fp = btn('False Positive (F)', 'danger')
        self.button_unc = btn('? Uncertain (U)', 'warning')
        self.button_bad_weather = btn('Bad Weather (W)', 'warning')

        self.slider_db = widgets.IntSlider(
            value=self.dynamic_range, min=20, max=100, step=5,
            description='dB Range:', layout=widgets.Layout(width='300px')
        )
        self.button_refresh = btn('Refresh Spectrogram', 'info', '180px')

        self.button_play_left = btn('Play Left', 'info', '120px')
        self.button_play_right = btn('Play Right', 'info', '120px')
        self.button_play_both = btn('Play Both', 'info', '120px')
        self.button_stop = btn('Stop', '', '100px')

        self.button_back = btn('Back (B)', '', '150px')
        self.button_skip = btn('Skip (S)', '', '150px')
        self.button_save = btn('Save Progress', 'primary', '150px')

        # Wire up callbacks
        self.button_tp.on_click(lambda b: self._record('T'))
        self.button_fp.on_click(lambda b: self._record('F'))
        self.button_unc.on_click(lambda b: self._record('U'))
        self.button_bad_weather.on_click(lambda b: self._record('W'))
        self.button_refresh.on_click(lambda b: self._refresh_spectrogram())
        self.button_play_left.on_click(lambda b: self._play_audio('left'))
        self.button_play_right.on_click(lambda b: self._play_audio('right'))
        self.button_play_both.on_click(lambda b: self._play_audio('both'))
        self.button_stop.on_click(lambda b: sd.stop())
        self.button_back.on_click(lambda b: self._go_back())
        self.button_skip.on_click(lambda b: self._skip())
        self.button_save.on_click(lambda b: self._save_progress())

        self.progress = widgets.IntProgress(
            value=len(self.results), min=0, max=self.total_detections,
            description='Progress:', bar_style='info',
            layout=widgets.Layout(width='600px')
        )
        self.status_text = widgets.HTML(value=self._get_status_html())
        self.output = widgets.Output()

    def _get_status_html(self):
        completed = len(self.results)
        remaining = self.total_detections - completed

        stats = ""
        if self.results:
            results_df = pd.DataFrame(self.results)
            tp = (results_df['validation'] == 'T').sum()
            fp = (results_df['validation'] == 'F').sum()
            unc = (results_df['validation'] == 'U').sum()
            weather = (results_df['validation'] == 'W').sum()
            missing = (results_df['validation'] == 'M').sum()
            stats = f"<b>TP:</b> {tp} | <b>FP:</b> {fp} | <b>Uncertain:</b> {unc} | <b>Bad Weather:</b> {weather}"
            if missing > 0:
                stats += f" | <b>Missing:</b> {missing}"

        return (
            f"<div style='font-size: 14px; padding: 10px; background-color: #f0f0f0; border-radius: 5px;'>"
            f"<b>Validation Progress:</b> {completed} / {self.total_detections} "
            f"({remaining} remaining)<br>{stats}</div>"
        )

    def _load_audio_stereo(self, row):
        """Load audio and split into left/right channels."""
        return load_audio_stereo(row['file_path'], row['start_time'], row['end_time'])

    def _plot_dual_spectrograms(self, audio_left, audio_right, sr, row):
        """Display vertically stacked spectrograms for left and right channels."""
        self.dynamic_range = self.slider_db.value
        detection_duration = row['end_time'] - row['start_time']
        fig_width = max(detection_duration * 2, 6)

        fig, (ax_left, ax_right) = plt.subplots(2, 1, figsize=(fig_width, 5), sharex=True)

        D_left = librosa.amplitude_to_db(
            np.abs(librosa.stft(audio_left, n_fft=512, hop_length=218)), ref=np.max
        )
        D_right = librosa.amplitude_to_db(
            np.abs(librosa.stft(audio_right, n_fft=512, hop_length=218)), ref=np.max
        )

        norm = PowerNorm(gamma=1, vmin=-self.dynamic_range, vmax=0)

        img_left = librosa.display.specshow(
            D_left, sr=sr, x_axis='time', y_axis='hz',
            hop_length=218, ax=ax_left, cmap=self.cmap, norm=norm
        )
        ax_left.set_ylim(1000, 10000)
        ax_left.set_title('Left Channel', fontsize=10)
        ax_left.set_xlabel('')

        img_right = librosa.display.specshow(
            D_right, sr=sr, x_axis='time', y_axis='hz',
            hop_length=218, ax=ax_right, cmap=self.cmap, norm=norm
        )
        ax_right.set_ylim(1000, 10000)
        ax_right.set_title('Right Channel', fontsize=10)

        fig.suptitle(
            f"Species: {row['species']} | Score: {row['confidence']:.3f} | "
            f"File: {Path(row['file_path']).name} | "
            f"Time: {row['start_time']:.1f}-{row['end_time']:.1f}s",
            fontsize=10
        )

        fig.colorbar(img_left, ax=ax_left, format='%+2.0f dB')
        fig.colorbar(img_right, ax=ax_right, format='%+2.0f dB')
        plt.tight_layout()
        plt.subplots_adjust(top=0.9)
        plt.show()

    def _show_detection(self):
        """Display the current detection."""
        with self.output:
            clear_output(wait=True)

            if self.current_idx >= len(self.detections):
                self._validation_complete()
                return

            row = self.detections.iloc[self.current_idx]

            if pd.isna(row['file_path']):
                display(widgets.HTML(
                    f"<div style='color: orange;'>Missing audio file for "
                    f"validation_id {row['validation_id']}. Auto-skipping...</div>"
                ))
                self._record('M')
                return

            audio_left, audio_right, sr = self._load_audio_stereo(row)

            if audio_left is None:
                display(widgets.HTML(
                    f"<div style='color: red;'>Error loading: {row['file_path']}<br>"
                    f"Use Skip (S) to move to next detection.</div>"
                ))
                return

            self.current_audio_left = audio_left
            self.current_audio_right = audio_right
            self.current_sr = sr

            self._plot_dual_spectrograms(audio_left, audio_right, sr, row)
            display(Audio(audio_left, rate=sr, autoplay=False))

    def _refresh_spectrogram(self):
        if self.current_idx >= len(self.detections):
            return
        row = self.detections.iloc[self.current_idx]
        with self.output:
            clear_output(wait=True)
            if self.current_audio_left is not None:
                self._plot_dual_spectrograms(
                    self.current_audio_left, self.current_audio_right,
                    self.current_sr, row
                )
                display(Audio(self.current_audio_left, rate=self.current_sr, autoplay=False))

    def _play_audio(self, channel):
        play_audio(channel, self.current_audio_left, self.current_audio_right, self.current_sr)

    def _record(self, validation_code):
        if self.current_idx >= len(self.detections):
            return
        row = self.detections.iloc[self.current_idx].to_dict()
        row['validation'] = validation_code
        row['validation_timestamp'] = datetime.now().isoformat()
        self.results.append(row)

        self.current_idx += 1
        self.progress.value = len(self.results)
        self.status_text.value = self._get_status_html()

        if len(self.results) % 20 == 0:
            self._save_progress(silent=True)

        self._show_detection()

    def _skip(self):
        if self.current_idx >= len(self.detections):
            return
        skipped = self.detections.iloc[[self.current_idx]]
        self.detections = pd.concat([
            self.detections.iloc[:self.current_idx],
            self.detections.iloc[self.current_idx+1:],
            skipped
        ]).reset_index(drop=True)
        self._show_detection()

    def _go_back(self):
        if len(self.results) == 0:
            with self.output:
                display(widgets.HTML("<div style='color: orange;'>Already at first detection</div>"))
            return
        self.results.pop()
        self.current_idx = max(0, self.current_idx - 1)
        self.progress.value = len(self.results)
        self.status_text.value = self._get_status_html()
        self._show_detection()

    def _save_progress(self, silent=False):
        if len(self.results) == 0:
            if not silent:
                with self.output:
                    display(widgets.HTML("<div style='color: orange;'>No validations to save yet</div>"))
            return
        df = pd.DataFrame(self.results)
        df.to_csv(self.progress_file, index=False)
        if not silent:
            with self.output:
                display(widgets.HTML(
                    f"<div style='color: green;'>Saved {len(self.results)} validations to {self.progress_file}</div>"
                ))

    def _validation_complete(self):
        df = pd.DataFrame(self.results)
        df.to_csv(self.results_file, index=False)

        tp = (df['validation'] == 'T').sum()
        fp = (df['validation'] == 'F').sum()
        unc = (df['validation'] == 'U').sum()
        weather = (df['validation'] == 'W').sum()
        missing = (df['validation'] == 'M').sum()

        display(widgets.HTML(f"""
        <div style='padding: 20px; background-color: #d4edda; border: 2px solid #28a745; border-radius: 10px;'>
            <h2 style='color: #155724;'>Validation Complete!</h2>
            <p style='font-size: 16px;'>
                <b>Total validated:</b> {len(df)}<br>
                <b>True Positives:</b> {tp} ({tp/len(df)*100:.1f}%)<br>
                <b>False Positives:</b> {fp} ({fp/len(df)*100:.1f}%)<br>
                <b>Uncertain:</b> {unc} ({unc/len(df)*100:.1f}%)<br>
                <b>Bad Weather:</b> {weather} ({weather/len(df)*100:.1f}%)<br>
                <b>Missing Files:</b> {missing} ({missing/len(df)*100:.1f}%)
            </p>
            <p><b>Results saved to:</b> {self.results_file}</p>
        </div>
        """))

        print("\nValidation summary by species:")
        print(df.groupby(['species', 'validation']).size().unstack(fill_value=0))

    def display(self):
        """Show the validation interface."""
        validation_buttons = widgets.HBox(
            [self.button_tp, self.button_fp, self.button_unc, self.button_bad_weather],
            layout=widgets.Layout(justify_content='center')
        )
        spec_controls = widgets.HBox(
            [self.slider_db, self.button_refresh],
            layout=widgets.Layout(justify_content='center')
        )
        audio_controls = widgets.HBox(
            [self.button_play_left, self.button_play_right, self.button_play_both, self.button_stop],
            layout=widgets.Layout(justify_content='center')
        )
        nav_controls = widgets.HBox(
            [self.button_back, self.button_skip, self.button_save],
            layout=widgets.Layout(justify_content='center')
        )

        display(self.status_text)
        display(self.progress)
        display(validation_buttons)
        display(spec_controls)
        display(audio_controls)
        display(nav_controls)
        display(widgets.HTML(
            "<div style='text-align: center; margin: 10px; color: #666;'>"
            "<b>Keyboard shortcuts:</b> T = True Positive | F = False Positive | "
            "U = Uncertain | W = Bad Weather | B = Back | S = Skip"
            "</div>"
        ))
        display(self.output)
        self._show_detection()

In [ ]:
# Load sample (or use validation_sample from previous cell)
# validation_sample = pd.read_csv(SAMPLE_FILE)

validator = ValidationInterface(
    validation_sample,
    progress_file=PROGRESS_FILE,
    results_file=RESULTS_FILE,
    colormap='bw'  # 'bw' for grayscale, 'color' for yellow-red
)
validator.display()

## 6. Review Results

After completing validation (or loading saved results), review the summary statistics.

In [ ]:
# Load final results
results = pd.read_csv(RESULTS_FILE)

print(f"Total validated: {len(results)}")
print(f"\nOverall breakdown:")
print(results['validation'].value_counts())

print(f"\nBy species:")
summary = results.groupby(['species', 'validation']).size().unstack(fill_value=0)
print(summary)

# Precision by species (TP / (TP + FP))
print(f"\nPrecision by species (excluding Uncertain/Weather/Missing):")
for species in summary.index:
    tp = summary.loc[species].get('T', 0)
    fp = summary.loc[species].get('F', 0)
    if tp + fp > 0:
        precision = tp / (tp + fp)
        print(f"  {species}: {precision:.3f} ({tp} TP, {fp} FP)")
    else:
        print(f"  {species}: no TP/FP results")